In [7]:
import pandas as pd

from src.features.nhanes_features import (
    create_basic_health_features,
    clean_features
)

In [8]:
import os

os.getcwd()

'/workspaces/adaptive-health-engine/notebooks'

In [9]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [10]:
import pandas as pd

from src.features.nhanes_features import (
    create_basic_health_features,
    clean_features
)

In [11]:
import pandas as pd

body = pd.read_sas(
    "../data/raw/nhanes/body.XPT"
)

demo = pd.read_sas(
    "../data/raw/nhanes/demographics.XPT"
)

print(body.shape)
print(demo.shape)

(8860, 22)
(11933, 27)


In [12]:
df = demo.merge(
    body,
    on="SEQN",
    how="inner"
)

print(df.shape)

(8860, 48)


In [13]:
df.columns.tolist()

['SEQN',
 'SDDSRVYR',
 'RIDSTATR',
 'RIAGENDR',
 'RIDAGEYR',
 'RIDAGEMN',
 'RIDRETH1',
 'RIDRETH3',
 'RIDEXMON',
 'RIDEXAGM',
 'DMQMILIZ',
 'DMDBORN4',
 'DMDYRUSR',
 'DMDEDUC2',
 'DMDMARTZ',
 'RIDEXPRG',
 'DMDHHSIZ',
 'DMDHRGND',
 'DMDHRAGZ',
 'DMDHREDZ',
 'DMDHRMAZ',
 'DMDHSEDZ',
 'WTINT2YR',
 'WTMEC2YR',
 'SDMVSTRA',
 'SDMVPSU',
 'INDFMPIR',
 'BMDSTATS',
 'BMXWT',
 'BMIWT',
 'BMXRECUM',
 'BMIRECUM',
 'BMXHEAD',
 'BMIHEAD',
 'BMXHT',
 'BMIHT',
 'BMXBMI',
 'BMDBMIC',
 'BMXLEG',
 'BMILEG',
 'BMXARML',
 'BMIARML',
 'BMXARMC',
 'BMIARMC',
 'BMXWAIST',
 'BMIWAIST',
 'BMXHIP',
 'BMIHIP']

In [14]:
health_features = create_basic_health_features(df)

health_features.head()

,SEQN,age,weight_kg,height_cm,bmi
0,130378.0,43.0,86.9,179.5,27.0
1,130379.0,66.0,101.8,174.2,33.5
2,130380.0,44.0,69.4,152.9,29.7
3,130381.0,5.0,34.3,120.1,23.8
4,130382.0,2.0,13.6,NaN,NaN


In [15]:
health_features["age"].describe()

count    8.860000e+03
mean     3.990542e+01
std      2.517546e+01
min      5.397605e-79
25%      1.500000e+01
50%      4.000000e+01
75%      6.300000e+01
max      8.000000e+01
Name: age, dtype: float64

In [16]:
adult_features = health_features[
    health_features["age"] >= 18
]

adult_features.shape


(6337, 5)

Raw NHANES participants:
8860

After adult filter (age ≥18):
6337 participants

Features:
4
- age
- weight_kg
- height_cm
- bmi

In [17]:
adult_features.isnull().sum()

SEQN           0
age            0
weight_kg     89
height_cm     75
bmi          102
dtype: int64

adult_features
6337 participants

Missing values:

age          0
weight_kg   89
height_cm   75
bmi         102

In [18]:
clean_features = adult_features.dropna()

clean_features.shape

(6235, 5)

Adult NHANES dataset

Before cleaning:
6337 participants

After removing missing measurements:
6235 participants

Features:
- age
- weight_kg
- height_cm
- bmi

In [19]:
clean_features.to_csv(
    "../data/processed/nhanes_health_features.csv",
    index=False
)

In [20]:
import os

os.path.exists("../data/processed/nhanes_health_features.csv")

True

In [21]:
import pandas as pd

test_load = pd.read_csv(
    "../data/processed/nhanes_health_features.csv"
)

test_load.head()

,SEQN,age,weight_kg,height_cm,bmi
0,130378.0,43.0,86.9,179.5,27.0
1,130379.0,66.0,101.8,174.2,33.5
2,130380.0,44.0,69.4,152.9,29.7
3,130386.0,34.0,90.6,173.3,30.2
4,130387.0,68.0,103.5,155.9,42.6


In [22]:
from src.models.clustering import create_clusters

print("clustering module works")

clustering module works


In [23]:
import pandas as pd

health_data = pd.read_csv(
    "../data/processed/nhanes_health_features.csv"
)

health_data.shape

(6235, 5)

Processed dataset loaded:

6235 participants
4 features:

- age
- weight_kg
- height_cm
- bmi

In [24]:
X = health_data[
    [
        "age",
        "weight_kg",
        "height_cm",
        "bmi"
    ]
]

X.head()

,age,weight_kg,height_cm,bmi
0,43.0,86.9,179.5,27.0
1,66.0,101.8,174.2,33.5
2,44.0,69.4,152.9,29.7
3,34.0,90.6,173.3,30.2
4,68.0,103.5,155.9,42.6


X matrix:

6235 participants

Features:
- age
- weight_kg
- height_cm
- bmi

In [25]:
from src.models.clustering import create_clusters

labels, model = create_clusters(
    X,
    n_clusters=3
)

labels[:10]

array([0, 2, 0, 0, 2, 2, 1, 2, 2, 1], dtype=int32)

Input:
6235 participants
4 physiological features

Model:
KMeans

Clusters:
3

Output:
cluster labels generated ✅

In [26]:
clustered_data = X.copy()

clustered_data["cluster"] = labels

clustered_data.head()

,age,weight_kg,height_cm,bmi,cluster
0,43.0,86.9,179.5,27.0,0
1,66.0,101.8,174.2,33.5,2
2,44.0,69.4,152.9,29.7,0
3,34.0,90.6,173.3,30.2,0
4,68.0,103.5,155.9,42.6,2


In [27]:
cluster_summary = clustered_data.groupby("cluster").mean()

cluster_summary

,age,weight_kg,height_cm,bmi
cluster,,,,
0,32.569064,72.092121,167.247387,25.800776
1,66.811214,73.950319,164.195067,27.472818
2,50.424528,113.839084,171.877763,38.837332


KMeans produced 3 physiological profiles:

Cluster 0:
Age:        32.6 years
Weight:     72.1 kg
Height:     167.2 cm
BMI:        25.8

Cluster 1:
Age:        66.8 years
Weight:     73.9 kg
Height:     164.2 cm
BMI:        27.5

Cluster 2:
Age:        50.4 years
Weight:     113.8 kg
Height:     171.9 cm
BMI:        38.8

In [28]:
clustered_data["cluster"].value_counts()

cluster
1    2818
0    1933
2    1484
Name: count, dtype: int64

Cluster sizes:

Cluster 1:
2818 participants (45.2%)

Cluster 0:
1933 participants (31.0%)

Cluster 2:
1484 participants (23.8%)

Current interpretation:

Cluster 0
≈ younger/lower weight phenotype

Cluster 1
≈ older/moderate BMI phenotype

Cluster 2
≈ high adiposity phenotype

But there is a major limitation:

Right now the clustering is driven almost entirely by:

age
height
weight
BMI

It has not yet learned behavior.

This is only the physiological layer.

The future adaptive model needs:

Physiology
+
Executive function
+
ADHD traits
+
Lifestyle constraints
+
Adherence patterns

        ↓

Personalized intervention strategy

In [29]:
clustered_data.to_csv(
    "../data/processed/nhanes_health_clusters.csv",
    index=False
)

In [30]:
import os

os.path.exists("../data/processed/nhanes_health_clusters.csv")

True

In [31]:
import pandas as pd

paq = pd.read_sas(
    "../data/raw/nhanes/PAQ_J.XPT"
)

paq.shape

(5856, 17)

In [32]:
paq.columns.tolist()

['SEQN',
 'PAQ605',
 'PAQ610',
 'PAD615',
 'PAQ620',
 'PAQ625',
 'PAD630',
 'PAQ635',
 'PAQ640',
 'PAD645',
 'PAQ650',
 'PAQ655',
 'PAD660',
 'PAQ665',
 'PAQ670',
 'PAD675',
 'PAD680']

In [33]:
import sys
import os

sys.path.append(os.path.abspath(".."))
from src.features.activity_features import (
    create_activity_features
)

print("activity feature module works")

activity feature module works


In [34]:
import pandas as pd

paq = pd.read_sas(
    "../data/raw/nhanes/PAQ_J.XPT"
)

activity_features = create_activity_features(paq)

activity_features.head()

,SEQN,sedentary_minutes,vigorous_activity,moderate_activity,work_activity,activity_score
0,93705.0,300.0,2.0,1.0,4.0,7.0
1,93706.0,240.0,2.0,1.0,4.0,7.0
2,93708.0,120.0,2.0,1.0,4.0,7.0
3,93709.0,600.0,2.0,2.0,3.0,7.0
4,93711.0,420.0,1.0,1.0,4.0,6.0


In [35]:
import pandas as pd

health_data = pd.read_csv(
    "../data/processed/nhanes_health_features.csv"
)

health_data.head()

,SEQN,age,weight_kg,height_cm,bmi
0,130378.0,43.0,86.9,179.5,27.0
1,130379.0,66.0,101.8,174.2,33.5
2,130380.0,44.0,69.4,152.9,29.7
3,130386.0,34.0,90.6,173.3,30.2
4,130387.0,68.0,103.5,155.9,42.6


In [36]:
import pandas as pd

from src.features.nhanes_features import (
    create_basic_health_features,
    clean_features
)

body = pd.read_sas(
    "../data/raw/nhanes/body.XPT"
)

health_features = create_basic_health_features(body)

health_features = clean_features(
    health_features
)

health_features.head()

KeyError: 'RIDAGEYR'

In [ ]:
print(body.columns.tolist()[:20])

['SEQN', 'BMDSTATS', 'BMXWT', 'BMIWT', 'BMXRECUM', 'BMIRECUM', 'BMXHEAD', 'BMIHEAD', 'BMXHT', 'BMIHT', 'BMXBMI', 'BMDBMIC', 'BMXLEG', 'BMILEG', 'BMXARML', 'BMIARML', 'BMXARMC', 'BMIARMC', 'BMXWAIST', 'BMIWAIST']


In [ ]:
demographics = pd.read_sas(
    "../data/raw/nhanes/demographics.XPT"
)

print(demographics.columns.tolist()[:20])

['SEQN', 'SDDSRVYR', 'RIDSTATR', 'RIAGENDR', 'RIDAGEYR', 'RIDAGEMN', 'RIDRETH1', 'RIDRETH3', 'RIDEXMON', 'RIDEXAGM', 'DMQMILIZ', 'DMDBORN4', 'DMDYRUSR', 'DMDEDUC2', 'DMDMARTZ', 'RIDEXPRG', 'DMDHHSIZ', 'DMDHRGND', 'DMDHRAGZ', 'DMDHREDZ']


In [ ]:
body = pd.read_sas(
    "../data/raw/nhanes/body.XPT"
)

demographics = pd.read_sas(
    "../data/raw/nhanes/demographics.XPT"
)

nhanes = demographics.merge(
    body,
    on="SEQN",
    how="inner"
)

nhanes.shape

(8860, 48)

In [ ]:
health_features = create_basic_health_features(
    nhanes
)

health_features = clean_features(
    health_features
)

health_features.head()

,SEQN,age,weight_kg,height_cm,bmi
0,130378.0,43.0,86.9,179.5,27.0
1,130379.0,66.0,101.8,174.2,33.5
2,130380.0,44.0,69.4,152.9,29.7
3,130381.0,5.0,34.3,120.1,23.8
5,130386.0,34.0,90.6,173.3,30.2


In [ ]:
health_behavior = health_features.merge(
    activity_features,
    on="SEQN",
    how="inner"
)

health_behavior.shape

(0, 10)

In [ ]:
print(health_features["SEQN"].head())
print(activity_features["SEQN"].head())

0    130378.0
1    130379.0
2    130380.0
3    130381.0
5    130386.0
Name: SEQN, dtype: float64
0    93705.0
1    93706.0
2    93708.0
3    93709.0
4    93711.0
Name: SEQN, dtype: float64


In [ ]:
body_j = pd.read_sas(
    "../data/raw/nhanes/body_J.XPT"
)

demo_j = pd.read_sas(
    "../data/raw/nhanes/demographics_J.XPT"
)

print(body_j["SEQN"].head())
print(demo_j["SEQN"].head())


0    93703.0
1    93704.0
2    93705.0
3    93706.0
4    93707.0
Name: SEQN, dtype: float64
0    93703.0
1    93704.0
2    93705.0
3    93706.0
4    93707.0
Name: SEQN, dtype: float64


In [ ]:
nhanes_j = demo_j.merge(
    body_j,
    on="SEQN",
    how="inner"
)

nhanes_j.shape

(8704, 66)

In [ ]:
health_features_j = create_basic_health_features(
    nhanes_j
)

health_features_j = clean_features(
    health_features_j
)

health_features_j.head()

,SEQN,age,weight_kg,height_cm,bmi
0,93703.0,2.0,13.7,88.6,17.5
1,93704.0,2.0,13.9,94.2,15.7
2,93705.0,66.0,79.5,158.3,31.7
3,93706.0,18.0,66.3,175.7,21.5
4,93707.0,13.0,45.4,158.4,18.1


In [ ]:
activity_features_j = create_activity_features(
    paq
)

health_behavior_j = health_features_j.merge(
    activity_features_j,
    on="SEQN",
    how="inner"
)

health_behavior_j.shape

(5434, 10)

health_behavior_j

Rows:
5434 participants

Columns:
10

Features:

SEQN
age
weight_kg
height_cm
bmi

+

sedentary_minutes
vigorous_activity
moderate_activity
work_activity
activity_score

In [ ]:
health_behavior_j.to_csv(
    "../data/processed/nhanes_adaptive_features.csv",
    index=False
)

In [ ]:
health_behavior_j.shape

(5434, 10)


Adaptive Health Engine v0.3

INPUT:
2017-2018 NHANES

DATA SOURCES:
✓ DEMO_J
✓ BMX_J
✓ PAQ_J


FEATURE SPACE:

Physiology:
- age
- weight_kg
- height_cm
- bmi

Behavior:
- sedentary_minutes
- vigorous_activity
- moderate_activity
- work_activity
- activity_score


OUTPUT:
5434 participant adaptive representations

In [ ]:
health_behavior_j.info()


<class 'pandas.DataFrame'>
RangeIndex: 5434 entries, 0 to 5433
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   SEQN               5434 non-null   float64
 1   age                5434 non-null   float64
 2   weight_kg          5434 non-null   float64
 3   height_cm          5434 non-null   float64
 4   bmi                5434 non-null   float64
 5   sedentary_minutes  5425 non-null   float64
 6   vigorous_activity  5434 non-null   float64
 7   moderate_activity  5434 non-null   float64
 8   work_activity      5434 non-null   float64
 9   activity_score     5434 non-null   float64
dtypes: float64(10)
memory usage: 424.7 KB


In [ ]:
from src.models.adaptive_clustering import (
    create_adaptive_clusters
)

print("adaptive clustering module works")

adaptive clustering module works


In [ ]:
from src.models.adaptive_clustering import (
    create_adaptive_clusters
)

adaptive_clusters, model, scaler = create_adaptive_clusters(
    health_behavior_j,
    n_clusters=4
)

adaptive_clusters.head()

,SEQN,age,weight_kg,height_cm,bmi,sedentary_minutes,vigorous_activity,moderate_activity,work_activity,activity_score,cluster
0,93705.0,66.0,79.5,158.3,31.7,300.0,2.0,1.0,4.0,7.0,0
1,93706.0,18.0,66.3,175.7,21.5,240.0,2.0,1.0,4.0,7.0,1
2,93708.0,66.0,53.5,150.2,23.7,120.0,2.0,1.0,4.0,7.0,0
3,93709.0,75.0,88.8,151.1,38.9,600.0,2.0,2.0,3.0,7.0,0
4,93711.0,56.0,62.1,170.6,21.3,420.0,1.0,1.0,4.0,6.0,1


In [ ]:
adaptive_clusters.groupby("cluster").mean(numeric_only=True)

,SEQN,age,weight_kg,height_cm,bmi,sedentary_minutes,vigorous_activity,moderate_activity,work_activity,activity_score
cluster,,,,,,,,,,
0,98286.980460,56.362452,71.194444,162.369004,27.053870,325.637026,1.981226,1.782759,3.692720,7.456705
1,98196.659645,39.092166,76.892298,168.848124,26.942989,314.307388,1.217248,1.211982,2.984200,5.413430
2,98240.700000,57.066667,81.676667,165.033333,30.120000,9999.000000,1.900000,1.800000,3.766667,7.466667
3,98345.251765,48.448627,112.234824,171.591843,38.356157,361.299213,1.921569,1.688627,3.076863,6.687059


In [ ]:
health_behavior_j["sedentary_minutes"].value_counts().tail()

sedentary_minutes
1140.0    1
35.0      1
6.0       1
8.0       1
50.0      1
Name: count, dtype: int64

In [ ]:
health_behavior_j["sedentary_minutes"].describe()
health_behavior_j[
    health_behavior_j["sedentary_minutes"] > 1000
]

,SEQN,age,weight_kg,height_cm,bmi,sedentary_minutes,vigorous_activity,moderate_activity,work_activity,activity_score
50,93780.0,46.0,87.5,161.7,33.5,1020.0,2.0,2.0,4.0,8.0
67,93805.0,64.0,89.9,162.9,33.9,9999.0,2.0,1.0,4.0,7.0
115,93887.0,63.0,78.4,174.2,25.8,9999.0,2.0,2.0,4.0,8.0
575,94619.0,61.0,80.9,174.3,26.6,9999.0,2.0,2.0,4.0,8.0
662,94775.0,26.0,81.1,159.3,32.0,1020.0,1.0,2.0,4.0,7.0
967,95275.0,41.0,55.4,160.9,21.4,1020.0,2.0,2.0,3.0,7.0
986,95315.0,63.0,109.0,155.6,45.0,1080.0,2.0,2.0,4.0,8.0
1104,95515.0,69.0,121.1,190.0,33.5,9999.0,1.0,2.0,11.0,14.0
1220,95690.0,54.0,75.1,169.4,26.2,1020.0,2.0,2.0,4.0,8.0
1319,95860.0,51.0,53.8,162.4,20.4,1320.0,2.0,2.0,11.0,15.0


In [ ]:
health_behavior_j["sedentary_minutes"].describe()

count    5.425000e+03
mean     3.843128e+02
std      7.440437e+02
min      5.397605e-79
25%      1.800000e+02
50%      3.000000e+02
75%      4.800000e+02
max      9.999000e+03
Name: sedentary_minutes, dtype: float64

In [ ]:
health_behavior_j[
    health_behavior_j["sedentary_minutes"] > 1000
]

,SEQN,age,weight_kg,height_cm,bmi,sedentary_minutes,vigorous_activity,moderate_activity,work_activity,activity_score
50,93780.0,46.0,87.5,161.7,33.5,1020.0,2.0,2.0,4.0,8.0
67,93805.0,64.0,89.9,162.9,33.9,9999.0,2.0,1.0,4.0,7.0
115,93887.0,63.0,78.4,174.2,25.8,9999.0,2.0,2.0,4.0,8.0
575,94619.0,61.0,80.9,174.3,26.6,9999.0,2.0,2.0,4.0,8.0
662,94775.0,26.0,81.1,159.3,32.0,1020.0,1.0,2.0,4.0,7.0
967,95275.0,41.0,55.4,160.9,21.4,1020.0,2.0,2.0,3.0,7.0
986,95315.0,63.0,109.0,155.6,45.0,1080.0,2.0,2.0,4.0,8.0
1104,95515.0,69.0,121.1,190.0,33.5,9999.0,1.0,2.0,11.0,14.0
1220,95690.0,54.0,75.1,169.4,26.2,1020.0,2.0,2.0,4.0,8.0
1319,95860.0,51.0,53.8,162.4,20.4,1320.0,2.0,2.0,11.0,15.0


In [ ]:
import sys
import os

sys.path.append(os.path.abspath(".."))
from src.features.activity_features import (
    create_activity_features,
    clean_activity_features
)

print("activity cleaning module works")

activity cleaning module works


In [ ]:
import pandas as pd

paq = pd.read_sas(
    "../data/raw/nhanes/PAQ_J.XPT"
)

paq.shape

(5856, 17)

In [ ]:
activity_features_j = create_activity_features(
    paq
)

activity_features_j = clean_activity_features(
    activity_features_j
)

activity_features_j["sedentary_minutes"].describe()

count     5790.0
unique      50.0
top        240.0
freq       824.0
Name: sedentary_minutes, dtype: float64

In [39]:
import pandas as pd

body_j = pd.read_sas(
    "../data/raw/nhanes/body_J.XPT"
)

demo_j = pd.read_sas(
    "../data/raw/nhanes/demographics_J.XPT"
)

nhanes_j = demo_j.merge(
    body_j,
    on="SEQN",
    how="inner"
)

nhanes_j.shape

(8704, 66)

In [40]:
health_features_j = create_basic_health_features(
    nhanes_j
)

health_features_j = clean_features(
    health_features_j
)

health_features_j.shape

(8005, 5)

In [41]:
health_behavior_j_clean = health_features_j.merge(
    activity_features_j,
    on="SEQN",
    how="inner"
)

health_behavior_j_clean.shape

(5434, 10)

In [42]:
from src.models.adaptive_clustering import (
    create_adaptive_clusters
)

adaptive_clusters_clean, model, scaler = create_adaptive_clusters(
    health_behavior_j_clean,
    n_clusters=4
)

adaptive_clusters_clean.head()

,SEQN,age,weight_kg,height_cm,bmi,sedentary_minutes,vigorous_activity,moderate_activity,work_activity,activity_score,cluster
0,93705.0,66.0,79.5,158.3,31.7,300.0,2.0,1.0,4.0,7.0,1
1,93706.0,18.0,66.3,175.7,21.5,240.0,2.0,1.0,4.0,7.0,3
2,93708.0,66.0,53.5,150.2,23.7,120.0,2.0,1.0,4.0,7.0,1
3,93709.0,75.0,88.8,151.1,38.9,600.0,2.0,2.0,3.0,7.0,1
4,93711.0,56.0,62.1,170.6,21.3,420.0,1.0,1.0,4.0,6.0,2


In [43]:
adaptive_clusters_clean.groupby(
    "cluster"
).mean(numeric_only=True)

,SEQN,age,weight_kg,height_cm,bmi,vigorous_activity,moderate_activity,work_activity,activity_score
cluster,,,,,,,,,
0,98379.067275,49.069555,118.478335,170.545952,40.940251,1.908780,1.759407,3.423033,7.091220
1,98273.891423,56.756387,71.117062,161.873358,27.172856,1.984489,1.807482,3.881843,7.673814
2,98191.885901,38.083130,78.707905,169.106846,27.453790,1.000000,1.259169,3.143439,5.402608
3,98287.263620,49.032513,80.665993,168.809051,28.287698,2.000000,1.450791,2.494728,5.945518


Adaptive Health Phenotypes (v0.4)
🟠 Cluster 0 — High adiposity profile
Participants: 49 years old average

Weight:       118.5 kg
BMI:          40.9
Activity:     relatively high
Activity score: 7.1

Interpretation:

High body mass but not completely inactive.

This is interesting because the model detected:

severe obesity
preserved activity behaviors

Actionable implication:

Focus on body composition, nutrition adherence, resistance training, and metabolic monitoring — not simply "exercise more."

Potential intervention:

protein-focused diet
resistance training
gradual calorie deficit
waist/BMI tracking
🟢 Cluster 1 — Active moderate-risk profile
Age:          56.8
Weight:       71.1 kg
BMI:          27.2
Activity:     high
Activity score: 7.7

Interpretation:

This is a relatively healthy aging phenotype.

Characteristics:

overweight BMI range
good activity behavior
lower weight

Actionable implication:

Preserve current behaviors and prevent future decline.

Potential intervention:

maintain activity
strength training
monitor metabolic markers
🔵 Cluster 2 — Younger lower-activity profile
Age:          38.1
Weight:       78.7 kg
BMI:          27.5
Activity score: 5.4

Interpretation:

This is the most behaviorally concerning group despite not having the highest BMI.

Characteristics:

younger
overweight
lower activity
lower exercise participation

Actionable implication:

Prevention opportunity.

Potential intervention:

increase daily movement
reduce sedentary time
build exercise habit formation

This is exactly the type of group an adaptive system should detect early.

🟣 Cluster 3 — Intermediate-risk mixed profile
Age:          49.0
Weight:       80.7 kg
BMI:          28.3
Activity score: 5.9

Interpretation:

Moderate BMI with reduced activity.

Actionable implication:

Lifestyle optimization before progression to higher-risk phenotype.

Potential intervention:

increase moderate activity
improve consistency
monitor weight trajectory

In [44]:
adaptive_clusters_clean.to_csv(
    "../data/processed/nhanes_adaptive_clusters.csv",
    index=False
)


NHANES 2017-2018

DEMO_J
  +
BMX_J
  +
PAQ_J

      ↓

Feature Engineering

      ↓

Adaptive Health Dataset
(5434 participants)

      ↓

KMeans Phenotype Discovery

      ↓

4 Health Behavior Phenotypes

      ↓

Saved ML output

In [53]:
from src.recommendations.intervention_engine import (
    generate_recommendation
)

print("intervention engine works")

intervention engine works


In [46]:
cluster_0_profile = (
    adaptive_clusters_clean
    .groupby("cluster")
    .mean(numeric_only=True)
    .loc[0]
)

generate_recommendation(cluster_0_profile)

['Prioritize weight reduction and metabolic health.']

In [54]:
from src.recommendations.intervention_engine import (
    generate_recommendation
)

In [55]:
cluster_0_profile = (
    adaptive_clusters_clean
    .groupby("cluster")
    .mean(numeric_only=True)
    .loc[0]
)

generate_recommendation(cluster_0_profile)

['Prioritize weight reduction and metabolic health.']

In [56]:
from importlib import reload
import src.recommendations.intervention_engine as ie

reload(ie)

generate_recommendation = ie.generate_recommendation

In [57]:
cluster_0_profile = (
    adaptive_clusters_clean
    .groupby("cluster")
    .mean(numeric_only=True)
    .loc[0]
)

generate_recommendation(cluster_0_profile)

['High adiposity phenotype: prioritize sustainable weight reduction and metabolic health.',
 'Preserve muscle mass through resistance training and adequate protein intake.']

In [58]:
from src.recommendations.profile_generator import (
    generate_health_profile
)

cluster_profiles = (
    adaptive_clusters_clean
    .groupby("cluster")
    .mean(numeric_only=True)
)

profile = generate_health_profile(
    cluster_profiles.loc[0],
    0
)

profile

{'phenotype_id': 0,
 'age': np.float64(49.1),
 'bmi': np.float64(40.9),
 'weight_kg': np.float64(118.5),
 'activity_score': np.float64(7.1)}

In [59]:
from src.recommendations.report_generator import (
    generate_full_report
)

cluster_profiles = (
    adaptive_clusters_clean
    .groupby("cluster")
    .mean(numeric_only=True)
)

report = generate_full_report(
    cluster_profiles.loc[0],
    0
)

report

{'profile': {'phenotype_id': 0,
  'age': np.float64(49.1),
  'bmi': np.float64(40.9),
  'weight_kg': np.float64(118.5),
  'activity_score': np.float64(7.1)},
 'recommendations': ['High adiposity phenotype: prioritize sustainable weight reduction and metabolic health.',
  'Preserve muscle mass through resistance training and adequate protein intake.']}

In [60]:
from importlib import reload

import src.recommendations.profile_generator as pg
import src.recommendations.report_generator as rg

reload(pg)
reload(rg)

generate_full_report = rg.generate_full_report

In [61]:
report = generate_full_report(
    cluster_profiles.loc[0],
    0
)

report

{'profile': {'phenotype_id': 0,
  'age': 49.1,
  'bmi': 40.9,
  'weight_kg': 118.5,
  'activity_score': 7.1},
 'recommendations': ['High adiposity phenotype: prioritize sustainable weight reduction and metabolic health.',
  'Preserve muscle mass through resistance training and adequate protein intake.']}

In [62]:
from src.inference.phenotype_predictor import (
    prepare_individual_features
)

print("phenotype predictor works")

phenotype predictor works


In [63]:
from src.models.adaptive_clustering import (
    save_adaptive_model
)

save_adaptive_model(
    model,
    scaler
)

print("adaptive model saved")

ImportError: cannot import name 'save_adaptive_model' from 'src.models.adaptive_clustering' (/workspaces/adaptive-health-engine/src/models/adaptive_clustering.py)

In [64]:
from importlib import reload

import src.models.adaptive_clustering as ac

reload(ac)

save_adaptive_model = ac.save_adaptive_model

In [65]:
save_adaptive_model(
    model,
    scaler
)

print("adaptive model saved")

FileNotFoundError: [Errno 2] No such file or directory: 'src/models/artifacts'

In [66]:
from importlib import reload
import src.models.adaptive_clustering as ac

reload(ac)

save_adaptive_model = ac.save_adaptive_model

In [67]:
save_adaptive_model(
    model,
    scaler
)

print("adaptive model saved")

adaptive model saved


In [68]:
import os

print(os.getcwd())

/workspaces/adaptive-health-engine/notebooks


In [69]:
import os

print(os.listdir())

['02_feature_engineering.ipynb', '01_dataset_exploration.ipynb', 'src']


In [70]:
from importlib import reload

import src.models.adaptive_clustering as ac

reload(ac)

save_adaptive_model = ac.save_adaptive_model

In [71]:
save_adaptive_model(
    model,
    scaler
)

print("adaptive model saved")

adaptive model saved


In [72]:
import os

print(os.listdir("../src/models"))

['artifacts', 'clustering.py', 'adaptive_clustering.py', '__pycache__']


In [73]:
print(os.listdir("../src/models/artifacts"))

['adaptive_scaler.pkl', 'adaptive_cluster_model.pkl']


In [74]:
from src.inference.phenotype_predictor import (
    predict_phenotype
)

cluster = predict_phenotype(
    age=28,
    weight_kg=122,
    height_cm=183,
    activity_score=5
)

cluster

ImportError: cannot import name 'predict_phenotype' from 'src.inference.phenotype_predictor' (/workspaces/adaptive-health-engine/src/inference/phenotype_predictor.py)

In [75]:
from importlib import reload

import src.inference.phenotype_predictor as pp

reload(pp)

predict_phenotype = pp.predict_phenotype

In [76]:
cluster = predict_phenotype(
    age=28,
    weight_kg=122,
    height_cm=183,
    activity_score=5
)

cluster

ValueError: The feature names should match those that were passed during fit.
Feature names seen at fit time, yet now missing:
- moderate_activity
- sedentary_minutes
- vigorous_activity
- work_activity


In [77]:
from importlib import reload

import src.inference.phenotype_predictor as pp

reload(pp)

predict_phenotype = pp.predict_phenotype

In [78]:
cluster = predict_phenotype(
    age=28,
    weight_kg=122,
    height_cm=183,
    activity_score=5
)

cluster

2

In [79]:
from src.inference.health_predictor import (
    predict_health_profile
)

print("health predictor works")

health predictor works


In [80]:
result = predict_health_profile(
    age=28,
    weight_kg=122,
    height_cm=183,
    activity_score=5
)

result

FileNotFoundError: [Errno 2] No such file or directory: 'src/models/artifacts/adaptive_cluster_model.pkl'

In [81]:
from importlib import reload

import src.models.adaptive_clustering as ac

reload(ac)

<module 'src.models.adaptive_clustering' from '/workspaces/adaptive-health-engine/src/models/adaptive_clustering.py'>

In [82]:
import src.inference.phenotype_predictor as pp
reload(pp)

import src.inference.health_predictor as hp
reload(hp)

predict_health_profile = hp.predict_health_profile

In [83]:
result = predict_health_profile(
    age=28,
    weight_kg=122,
    height_cm=183,
    activity_score=5
)

result

{'profile': {'phenotype_id': 2,
  'age': 28.0,
  'bmi': 36.4,
  'weight_kg': 122.0,
  'activity_score': 5.0},
 'recommendations': ['High adiposity phenotype: prioritize sustainable weight reduction and metabolic health.',
  'Preserve muscle mass through resistance training and adequate protein intake.',
  'Low activity phenotype: gradually increase daily movement and structured exercise.']}

In [84]:
adaptive_clusters_clean.groupby("cluster").mean()

,SEQN,age,weight_kg,height_cm,bmi,sedentary_minutes,vigorous_activity,moderate_activity,work_activity,activity_score
cluster,,,,,,,,,,
0,98379.067275,49.069555,118.478335,170.545952,40.940251,422.871116,1.908780,1.759407,3.423033,7.091220
1,98273.891423,56.756387,71.117062,161.873358,27.172856,327.030079,1.984489,1.807482,3.881843,7.673814
2,98191.885901,38.083130,78.707905,169.106846,27.453790,326.359836,1.000000,1.259169,3.143439,5.402608
3,98287.263620,49.032513,80.665993,168.809051,28.287698,258.223111,2.000000,1.450791,2.494728,5.945518


In [85]:
from importlib import reload

import src.inference.health_predictor as hp

reload(hp)

predict_health_profile = hp.predict_health_profile

In [86]:
result = predict_health_profile(
    age=28,
    weight_kg=122,
    height_cm=183,
    activity_score=5
)

result

{'profile': {'phenotype_id': 2,
  'age': 28.0,
  'bmi': 36.4,
  'weight_kg': 122.0,
  'activity_score': 5.0,
  'phenotype_name': 'Low activity lifestyle phenotype',
  'phenotype_description': 'Characterized by reduced activity patterns with moderate adiposity, benefiting from increased movement.'},
 'recommendations': ['High adiposity phenotype: prioritize sustainable weight reduction and metabolic health.',
  'Preserve muscle mass through resistance training and adequate protein intake.',
  'Low activity phenotype: gradually increase daily movement and structured exercise.']}

In [87]:
from importlib import reload

import src.inference.health_predictor as hp

reload(hp)

predict_health_profile = hp.predict_health_profile

In [88]:
result = predict_health_profile(
    age=28,
    weight_kg=122,
    height_cm=183,
    activity_score=5
)

result

{'profile': {'phenotype_id': 2,
  'age': 28.0,
  'bmi': 36.4,
  'weight_kg': 122.0,
  'activity_score': 5.0,
  'phenotype_name': 'Low activity lifestyle phenotype',
  'phenotype_description': 'Characterized by reduced activity patterns with moderate adiposity, benefiting from increased movement.'},
 'recommendations': ['High adiposity phenotype: prioritize sustainable weight reduction and metabolic health.',
  'Preserve muscle mass through resistance training and adequate protein intake.',
  'Low activity phenotype: gradually increase daily movement and structured exercise.']}